In [ ]:
# ============================================================
#  RAG News Search Bot
#  Author  : Narjes
#  Stack   : LangChain · PyMuPDF · ChromaDB · phi3 (Ollama) · Gradio
#  Version : 1.0
# ============================================================

# ── Standard Library ────────────────────────────────────────
import os
import logging

# ── LangChain ───────────────────────────────────────────────
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_community.document_loaders.text import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.llms import Ollama
from langchain_core.prompts import PromptTemplate

# ── UI ──────────────────────────────────────────────────────
import gradio as gr


# ════════════════════════════════════════════════════════════
#  CONFIGURATION  ← only change these if needed
# ════════════════════════════════════════════════════════════

NEWS_DIR        = "./news_articles"   # put your .txt / .pdf news files here
CHROMA_DIR      = "./chroma_news_db"
EMBEDDING_MODEL = "all-MiniLM-L6-v2"
LLM_MODEL       = "phi3"
CHUNK_SIZE      = 500
CHUNK_OVERLAP   = 100
TOP_K           = 4

PROMPT_TEMPLATE = """
You are a helpful news assistant.

Using ONLY the news articles provided below, answer the user's question.
- Summarize clearly and concisely.
- If the topic is not covered in the articles, say: "I don't have news about that."
- Always mention which article the information comes from if possible.

News Context:
{context}

Question:
{question}

Answer:
"""


# ════════════════════════════════════════════════════════════
#  LOGGING
# ════════════════════════════════════════════════════════════

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
)
log = logging.getLogger(__name__)


# ════════════════════════════════════════════════════════════
#  PIPELINE SETUP
# ════════════════════════════════════════════════════════════

def load_documents(folder: str):
    """Load all .txt and .pdf files from a folder."""
    docs = []
    for filename in os.listdir(folder):
        filepath = os.path.join(folder, filename)
        if filename.endswith(".pdf"):
            loader = PyMuPDFLoader(filepath)
            docs.extend(loader.load())
            log.info("Loaded PDF: %s", filename)
        elif filename.endswith(".txt"):
            loader = TextLoader(filepath, encoding="utf-8")
            docs.extend(loader.load())
            log.info("Loaded TXT: %s", filename)
    log.info("Total documents loaded: %d", len(docs))
    return docs


def split_documents(documents):
    """Split documents into overlapping chunks for retrieval."""
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP,
    )
    chunks = splitter.split_documents(documents)
    log.info("Created %d chunks", len(chunks))
    return chunks


def build_vectorstore(chunks, embeddings):
    """Load existing ChromaDB or build a new one."""
    if os.path.exists(CHROMA_DIR):
        log.info("Loading existing vector store from '%s'", CHROMA_DIR)
        return Chroma(persist_directory=CHROMA_DIR, embedding_function=embeddings)
    log.info("Building new vector store at '%s'", CHROMA_DIR)
    return Chroma.from_documents(chunks, embeddings, persist_directory=CHROMA_DIR)


def setup_pipeline():
    """Full pipeline initialisation. Returns (retriever, llm, prompt)."""
    documents   = load_documents(NEWS_DIR)
    chunks      = split_documents(documents)
    embeddings  = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL)
    vectorstore = build_vectorstore(chunks, embeddings)
    llm         = Ollama(model=LLM_MODEL)
    retriever   = vectorstore.as_retriever(
        search_type="similarity",
        search_kwargs={"k": TOP_K},
    )
    prompt = PromptTemplate(
        input_variables=["context", "question"],
        template=PROMPT_TEMPLATE,
    )
    log.info("News Bot pipeline ready.")
    return retriever, llm, prompt

def why_it_matters(title, category):
    if category == "Economy":
        return "This may impact markets, jobs, or cost of living."
    elif category == "Tech":
        return "This could influence innovation and future technology trends."
    elif category == "Politics":
        return "This may affect policy decisions and public opinion."
    else:
        return "This is relevant for understanding current global events."
    
def classify_topic(text):
    text = text.lower()

    if any(word in text for word in ["ai", "software", "tech", "startup"]):
        return "Tech"
    elif any(word in text for word in ["election", "government", "policy"]):
        return "Politics"
    elif any(word in text for word in ["market", "inflation", "economy"]):
        return "Economy"
    else:
        return "General"
    
def remove_duplicates(articles):
    seen = set()
    unique = []

    for article in articles:
        title = article["title"]
        if title not in seen:
            unique.append(article)
            seen.add(title)

    return unique

# ════════════════════════════════════════════════════════════
#  INITIALISE
# ════════════════════════════════════════════════════════════

retriever, llm, prompt = setup_pipeline()


# ════════════════════════════════════════════════════════════
#  CORE Q&A FUNCTION
# ════════════════════════════════════════════════════════════

def search_news(question: str) -> str:
    """Retrieve relevant news chunks and return an LLM-generated summary."""
    if not question.strip():
        return "⚠️  Please enter a question."

    docs    = retriever.invoke(question)
    context = "\n\n".join(doc.page_content for doc in docs)

    prompt_text = prompt.format(context=context, question=question)
    answer      = llm.invoke(prompt_text)

    source_lines = ["\n\n─── Sources ───────────────────────────────"]
    for i, doc in enumerate(docs, start=1):
        source = doc.metadata.get("source", "Unknown")
        excerpt = doc.page_content[:200].strip().replace("\n", " ")
        source_lines.append(f"\n[{i}] 📰 {os.path.basename(source)}: {excerpt}…")

    return str(answer) + "\n".join(source_lines)


# ════════════════════════════════════════════════════════════
#  GRADIO UI
# ════════════════════════════════════════════════════════════

def build_ui() -> gr.Interface:
    return gr.Interface(
        fn=search_news,
        inputs=gr.Textbox(
            label="Your News Query",
            placeholder="e.g. What's happening in the US? Any news about AI layoffs?",
            lines=2,
        ),
        outputs=gr.Textbox(
            label="News Summary",
            lines=12,
        ),
        title="📰 RAG News Search Bot",
        description=(
            "Ask anything about the loaded news articles.\n"
            f"**Model:** {LLM_MODEL} · **Embeddings:** {EMBEDDING_MODEL} · **Vector DB:** ChromaDB\n"
            f"**News folder:** `{NEWS_DIR}` — drop your .txt or .pdf news files there."
        ),
        flagging_mode="never",
    )


# ════════════════════════════════════════════════════════════
#  ENTRY POINT
# ════════════════════════════════════════════════════════════

if __name__ == "__main__":
    log.info("Launching News Bot → http://127.0.0.1:7860")
    ui = build_ui()
    ui.launch(share=False, inline=True)

2026-04-09 18:25:58,745 [INFO] Loaded TXT: article 1.txt
2026-04-09 18:25:58,749 [INFO] Loaded TXT: article 2.txt
2026-04-09 18:25:58,753 [INFO] Loaded TXT: article 3.txt
2026-04-09 18:25:58,756 [INFO] Loaded TXT: article 4.txt
2026-04-09 18:25:58,758 [INFO] Loaded TXT: article 5.txt
2026-04-09 18:25:58,760 [INFO] Total documents loaded: 5
2026-04-09 18:25:58,766 [INFO] Created 14 chunks
2026-04-09 18:25:58,771 [INFO] Use pytorch device_name: cpu
2026-04-09 18:25:58,774 [INFO] Load pretrained SentenceTransformer: all-MiniLM-L6-v2
2026-04-09 18:25:59,374 [INFO] HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-04-09 18:25:59,453 [INFO] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/modules.json "HTTP/1.1 200 OK"
2026-04-09 18:25:59,641 [INFO] HTTP Request: HEAD https://huggingface.co/sentence-t

* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.


2026-04-09 18:26:04,056 [INFO] HTTP Request: HEAD https://huggingface.co/api/telemetry/https%3A/api.gradio.app/gradio-launched-telemetry "HTTP/1.1 200 OK"
2026-04-09 18:26:04,249 [INFO] HTTP Request: GET https://api.gradio.app/pkg-version "HTTP/1.1 200 OK"
